# Tarea 1

Hugo André Meza Fierros
A00841695


In [1]:
import pulp

## Problema 1

La administración de un viñedo desea combinar cuatro cosechas distintas para producir tres tipos de vinos en forma combinada. Las existencias de las cosechas y los precios de venta de los vinos combinados se muestran en la siguiente tabla, junto con ciertas restricciones sobre los porcentajes incluidos en la composición de las tres mezclas. En particular, las cosechas 2 y 3 en conjunto deberán constituir cuando menos 75% de la mezcla A y
cuando menos 35% de la mezcla C. Además, la mezcla A deberá contener cuando menos 8% de la cosecha 4, mientras que la mezcla B deberá contener por lo menos 10% de la cosecha 2 y a lo sumo 35% de la cosecha 4. Se podrá vender cualquier cantidad que se elabore de las mezclas A, B y C. Formule un modelo de PL que aproveche en la mejor forma las cosechas disponibles y resuélvalo.

| Mezcla | Cosecha 1 | Cosecha 2 | Cosecha 3 | Cosecha 4 | Precio de venta por galón |
| :---: | :--- | :--- | :--- | :--- | :---: |
| **A** | No hay restricción | Cuando menos 75% en cualquier proporción | Cuando menos 75% en cualquier proporción | Cuando menos 8% | 80 |
| **B** | No hay restricción | Cuando menos 10% | No hay restricción | Cuando mucho 35% | 50 |
| **C** | No hay restricción | Cuando menos 35% en cualquier proporción | Cuando menos 35% en cualquier proporción | No hay restricción | 35 |
| **Existencia (gal)** | 130 | 200 | 150 | 350 | --- |


### Variables de decisión

$I = \{1, 2, 3\}$
(Mezcla A, B, C)

$J = \{1, 2, 3, 4\}$
(Cosechas)

$x_{ij}: \text{Cantidad de galones de la cosecha } j \text{ asignados a la mezcla } i$

### Función objetivo

$\text{max } z = 80 (\sum_{j=1}^4 x_{1j}) + 50 \sum_{j=1}^4 x_{2j})+ 35 (\sum_{j=1}^4 x_{3j})$

### S.A: 

Existencia en galones:

$\quad x_{11} + x_{21} + x_{31} \leq 130$

$\quad x_{12} + x_{22} + x_{32} \leq 200$

$\quad x_{13} + x_{23} + x_{33} \leq 150$

$\quad x_{14} + x_{24} + x_{34} \leq 350$

Mezcla A:

$\quad x_{12} + x_{13} \geq 0.75 (\sum_{j=1}^4 x_{1j})$

$\quad x_{14} \geq 0.08 (\sum_{j=1}^4 x_{1j})$

Mezcla B:

$\quad x_{22} \geq 0.1 (\sum_{j=1}^4 x_{2j})$

$\quad x_{24} \leq 0.35 (\sum_{j=1}^4 x_{2j})$

Mezcla C:

$\quad x_{32} + x_{33} \geq 0.35 (\sum_{j=1}^4 x_{3j})$

No negatividad:

$\quad x_{ij} \geq 0, \forall i, j $

### Solución (Código)

In [2]:
I = [1, 2, 3] # Mezclas: 1=A, 2=B, 3=C
J = [1, 2, 3, 4] # Cosechas

precio = {1: 80, 2: 50, 3: 35}
existencia = {1: 130, 2: 200, 3: 150, 4: 350}

model1 = pulp.LpProblem("Vinedo", pulp.LpMaximize)

# Variables de decisión (Cantidad de cosecha j en mezcla i)
x = {(i, j): pulp.LpVariable(f"Cosecha_{j}_en_{i}", lowBound = 0) for i in I for j in J}

# Función objetivo
model1 += pulp.lpSum(precio[i] * pulp.lpSum(x[i, j] for j in J) for i in I)

# Restricciones

#Existencia
for j in J:
    model1 += pulp.lpSum(x[i, j] for i in I) <= existencia[j], f"Existencia_cosecha_{j}"

# Mezcla A:
model1 += x[1,2] + x[1,3] >= 0.75 * pulp.lpSum(x[1, j] for j in J), "Cosecha_2y3_MezclaA"
model1 += x[1, 4] >= 0.08 * pulp.lpSum(x[1, j] for j in J), "Cosecha_4_MezclaA"

# Mezcla B:
model1 += x[2, 2] >= 0.1 * pulp.lpSum(x[2, j] for j in J), "Cosecha_2_MezclaB"
model1 +=  x[2, 4] <= 0.35 * pulp.lpSum(x[2, j] for j in J), "Cosecha_4_MezclaB"

# Mezcla C:
model1 += x[3, 2] + x[3, 3] >= 0.35 * pulp.lpSum(x[3, j] for j in J), "Cosecha_2y3_MezclaC"

# Resolver el modelo
model1.solve()

# Resultados
print(f"Status: {pulp.LpStatus[model1.status]}")
print(f"Result: {pulp.value(model1.objective):.2f}")
      
nombres = {1: "A", 2: "B", 3: "C"}
print("Asignación (galones):")
print(f"{'Mezcla':<8}{'Cos.1':>10}{'Cos.2':>10}{'Cos.3':>10}{'Cos.4':>10}{'Total':>10}")
for i in I:
    fila = [pulp.value(x[i, j]) for j in J]
    total = sum(fila)
    print(f"{nombres[i]:<8}" + "".join(f"{v:>10.2f}" for v in fila) + f"{total:>10.2f}")
 
print("\nUso por cosecha:")
for j in J:
    usado = sum(pulp.value(x[i, j]) for i in I)
    print(f"  Cosecha {j}: {usado:.2f} / {existencia[j]} galones")

Status: Optimal
Result: 46630.30
Asignación (galones):
Mezcla       Cos.1     Cos.2     Cos.3     Cos.4     Total
A             0.00    176.36    150.00    108.79    435.15
B           130.00     23.64      0.00     82.73    236.36
C             0.00      0.00      0.00      0.00      0.00

Uso por cosecha:
  Cosecha 1: 130.00 / 130 galones
  Cosecha 2: 200.00 / 200 galones
  Cosecha 3: 150.00 / 150 galones
  Cosecha 4: 191.52 / 350 galones


## Problema 2
Un hospital emplea voluntarios para atender la recepción entre las 8:00 am y las 10:00 pm. Cada voluntario trabaja 3 horas consecutivas, excepto los que entran a las 8:00 pm, que solo trabajan 2 horas. Una aproximación a la necesidad mínima de voluntarios es por medio de una función escalonada en intervalos de dos horas, los cuales se inician a las 8:00 am como 4, 6, 8, 6, 4, 6 y 8. Como la mayoría de los voluntarios son pensionados, están
dispuestos a ofrecer sus servicios a cualquier hora del día (8:00 am a 10:00 pm). Sin embargo, como la mayoría de las instituciones caritativas compiten por sus servicios, la cantidad requerida debe mantenerse lo más baja posible. Determine un programa óptimo de la hora de inicio de los voluntarios. 


### Variables de decisión

| Turno | Inicio | Fin |
| :---: | :---: | :---:|
| 1 | 8 | 11 |
| 2 | 9 | 12 |
| 3 | 10 | 13 |
| 4 | 11 | 14 |
| 5 | 12 | 15 |
| 6 | 13 | 16 |
| 7 | 14 | 17 |
| 8 | 15 | 18 |
| 9 | 16 | 19 |
| 10 | 17 | 20 |
| 11 | 18 | 21 |
| 12 | 19 | 22 |
| 13 | 20 | 22 |


$I = \{ 1, 2, ..., 13\}$
(Turnos)

$x_i: \text{Numero de voluntarios que inician al comienzo del bloque } i$

### Función objetivo

$\text{min } z = \sum_{i=1}^{13} x_i$

### S.A

Necesidades mínimas:

Bloque 1: 

$\quad x_1 \geq 4$

$\quad x_1 + x_2 \geq 4$

Bloque 2:

$\quad x_1 + x_2 + x_3 \geq 6$

$\quad x_2 + x_3 + x_4 \geq 6$

Bloque 3:

$\quad x_3 + x_4 + x_5 \geq 8$

$\quad x_4 + x_5 + x_6 \geq 8$


Bloque 4:

$\quad x_5 + x_6 + x_7 \geq 6$

$\quad x_6 + x_7 + x_8 \geq 6$


Bloque 5:

$\quad x_7 + x_8 + x_9 \geq 4$

$\quad x_8 + x_9 + x_{10} \geq 4$

Bloque 6:

$\quad x_9 + x_{10} + x_{11} \geq 6$

$\quad x_{10}+ x_{11} + x_{12} \geq 6$

Bloque 7: 

$\quad x_{11} + x_{12} + x_{13} \geq 8$

$\quad x_{12} + x_{13} \geq 8$

No negatividad:

$\quad x_{i} \geq 0, \forall i \in \{ 1, 2, 3,..., 13 \}$

### Solución (Código)

In [6]:
I = range(1, 14) # Bloques

necesidad = {1: 4, 2: 6, 3: 8, 4: 6, 5: 4, 6: 6, 7: 8}

horarios = {
    1: "8-9 am", 2: "9-10 am",  3: "10-11 am", 4: "11-12 pm",
    5: "12-13 pm",  6: "13-14 pm",   7: "14-15 pm", 8: "15-16 pm",
    9: "16-17 pm", 10: "17-18 pm", 11: "18-19 pm", 12: "19-20 pm",
    13: "20-21 pm", 14: "21-22 pm"
}

model2 = pulp.LpProblem("Hospital", pulp.LpMinimize)

# Variables de decisión (Cantidad de voluntarios al inicio del Bloque i)
x = {(i): pulp.LpVariable(f"Voluntarios_inicio_Bloque{i}", lowBound = 0, cat='Integer') for i in I}

# Función objetivo
model2 += pulp.lpSum(x[i] for i in I)

# Restricciones (Necesidades mínimas)
model2 += x[1] >= necesidad[1], "Turno_1"
model2 += x[1] + x[2] >= necesidad[1], "Turno_2"
model2 += x[1] + x[2] + x[3] >= necesidad[2], "Turno_3"
model2 += x[2] + x[3] + x[4] >= necesidad[2], "Turno_4"
model2 += x[3] + x[4] + x[5] >= necesidad[3], "Turno_5"
model2 += x[4] + x[5] + x[6] >= necesidad[3], "Turno_6"
model2 += x[5] + x[6] + x[7] >= necesidad[4], "Turno_7"
model2 += x[6] + x[7] + x[8] >= necesidad[4], "Turno_8"
model2 += x[7] + x[8] + x[9] >= necesidad[5], "Turno_9"
model2 += x[8] + x[9] + x[10] >= necesidad[5], "Turno_10"
model2 += x[9] + x[10] + x[11] >= necesidad[6], "Turno_11"
model2 += x[10] + x[11] + x[12] >= necesidad[6], "Turno_12"
model2 += x[11] + x[12] + x[13] >= necesidad[7], "Turno_13"
model2 += x[12] + x[13] >= necesidad[7], "Turno_14"

# Resolver el modelo
model2.solve()

# Resultados
print(f"Status: {pulp.LpStatus[model2.status]}")
print(f"Result: {pulp.value(model2.objective):.0f} voluntarios totales en el día\n")

# Tabla de voluntarios por turno de inicio
print(f"{'Inicio':<12}{'Voluntarios':>12}")
print("-" * 24)
for i in I:
    print(f"{horarios[i]:<12}{int(pulp.value(x[i])):>12}")

# Tabla de cobertura vs demanda (CORREGIDA)
print(f"\n{'Período':<12}{'Demanda':>10}{'Cubierto':>12}{'Holgura':>10}")
print("-" * 44)

# Reconstruimos la cobertura para reflejar turnos de 3 bloques
cobertura = {
    1: x[1],
    2: x[1] + x[2],
    3: x[1] + x[2] + x[3],
    4: x[2] + x[3] + x[4],
    5: x[3] + x[4] + x[5],
    6: x[4] + x[5] + x[6],
    7: x[5] + x[6] + x[7],
    8: x[6] + x[7] + x[8],
    9: x[7] + x[8] + x[9],
    10: x[8] + x[9] + x[10],
    11: x[9] + x[10] + x[11],
    12: x[10] + x[11] + x[12],
    13: x[11] + x[12] + x[13],
    14: x[12] + x[13],
}

for i in I:
    # Mapeamos los 14 bloques a los 7 periodos de demanda
    # El bloque 1 y 2 corresponden a la necesidad 1. El 3 y 4 a la 2, etc.
    indice_necesidad = (i + 1) // 2  
    demanda_actual = necesidad[indice_necesidad]
    
    cub = int(pulp.value(cobertura[i]))
    holgura = cub - demanda_actual
    
    print(f"{horarios[i]:<12}{demanda_actual:>10}{cub:>12}{holgura:>10}")

Status: Optimal
Result: 32 voluntarios totales en el día

Inicio       Voluntarios
------------------------
8-9 am                 4
9-10 am                0
10-11 am               2
11-12 pm               6
12-13 pm               0
13-14 pm               2
14-15 pm               4
15-16 pm               0
16-17 pm               0
17-18 pm               6
18-19 pm               0
19-20 pm               0
20-21 pm               8

Período        Demanda    Cubierto   Holgura
--------------------------------------------
8-9 am               4           4         0
9-10 am              4           4         0
10-11 am             6           6         0
11-12 pm             6           8         2
12-13 pm             8           8         0
13-14 pm             8           8         0
14-15 pm             6           6         0
15-16 pm             6           6         0
16-17 pm             4           4         0
17-18 pm             4           6         2
18-19 pm             6    

## Problema 3

Gaermont Paper fabrica y vende papel a clientes mayoristas. La compañía fabrica un rollo de papel “estándar” de 190 pulgadas de ancho. Sin embargo, no necesariamente todos los pedidos son para este ancho. Es frecuente que la compañía reciba pedidos para rollos más angostos. Para satisfacer esos pedidos, los rollos más angostos se cortan de los rollos estándar. El número de rollos que se fabrican de cada ancho no es una cantidad fija, si no que el cliente ha establecido cierta flexibilidad y ha solicitado un número mínimo y un máximo de rollos, como se muestra en la tabla. Gaermont desea minimizar el número de pulgadas de desperdicio que se produce al tratar de satisfacer este pedido. Plantee un modelo de programación lineal apropiado para el problema y resuélvalo, ¿cuál es la cantidad mínima de desperdicio encontrada?

| Ancho del rollo (pulgadas) | Cantidad mínima | Cantidad máxima |
| :---: | :---: | :---: |
| 80 | 2500 | 3000 |
| 70 | 3200 | 3800 |
| 60 | 3800 | 4500 |
| 50 | 7000 | 8000 |


### Variables de decisión



| Tipo | 50 | 60 | 70 | 80 | Total | Sobrante |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| 1 | 3 | - | - | - | 150 | 40 |
| 2 | 2 | 1 | - | - | 160 | 30 |
| 3 | 2 | - | 1 | - | 170 | 20 |
| 4 | 2 | - | - | 1 | 180 | 10 |
| 5 | 1 | 1 | 1 | - | 180 | 10 |
| 6 | 1 | 1 | - | 1 | 190 | 0 |
| 7 | 1 | 2 | - | - | 170 | 20 |
| 8 | 1 | - | 2 | - | 190 | 0 |
| 9 | - | 3 | - | - | 180 | 10 |
| 10 | - | - | - | 2 | 160 | 30 |
| 11 | - | - | 1 | 1 | 150 | 40 |
| 12 | - | 2 | 1 | - | 190 | 0 |

$x_i: \text{Numero de rollos cortados del tipo } i$

### Función Objetivo

$\text{min } z = 40x_1 + 30x_2 + 20x_3 + 10x_4 + 10x_5 + 20x_7 + 10x_9 + 30x_{10} + 40 x_{11} $

### S.A.

Rollos de 50:

$\quad 3x_1 + 2x_2 + 2x_3 + 2x_4 + x_5 + x_6 + x_7 + x_8 \geq 7000$

$\quad 3x_1 + 2x_2 + 2x_3 + 2x_4 + x_5 + x_6 + x_7 + x_8 \leq 8000$

Rollos de 60:

$\quad x_2 + x_5 + x_6 + 2x_7 + 3x_9 + 2x_{12} \geq 3800$

$\quad x_2 + x_5 + x_6 + 2x_7 + 3x_9 + 2x_{12} \leq 4500$

Rollos de 70:

$\quad x_3 + x_5 + 2x_8 + x_{11} + x_{12} \geq 3200$

$\quad x_3 + x_5 + 2x_8 + x_{11} + x_{12} \leq 3800$

Rollos de 80:

$\quad x_4 + x_6 + 2x_{10} + x_{11} \geq 2500$

$\quad x_4 + x_6 + 2x_{10} + x_{11} \leq 3000$

No negatividad:

$\quad x_{i} \geq 0, \forall i \in \{ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12 \}$

### Solución (Código)

In [4]:
# Anchos de los rollos angostos
anchos = [50, 60, 70, 80]
I = range(1, 13)

residuos = {1: 40, 2: 30, 3: 20, 4: 10, 5: 10, 6: 0,
            7: 20, 8: 0, 9: 10, 10: 30, 11: 40, 12: 0}

minimo = {50: 7000, 60: 3800, 70: 3200, 80: 2500}
maximo = {50: 8000, 60: 4500, 70: 3800, 80: 3000}

#Modelo
model3 = pulp.LpProblem("Rollos", pulp.LpMinimize)
# Variables de decisión
x = {i: pulp.LpVariable(f"Corte_Tipo_{i}", lowBound=0, cat="Integer") for i in I}


# Función objetivo
model3 += pulp.lpSum(residuos[i] * x[i] for i in I)

# Restricciones

# Necesidades de Rollos de 50
model3 += 3 * x[1] + 2 * x[2] + 2 * x[3] + 2 * x[4] + x[5] + x[6] + x[7] + x[8] >= minimo[50], "Minimo_50"
model3 += 3 * x[1] + 2 * x[2] + 2 * x[3] + 2 * x[4] + x[5] + x[6] + x[7] + x[8] <= maximo[50], "Maximo_50"

# Necesidades de Rollos de 60 
model3 += x[2] + x[5] + x[6] + 2 * x[7] + 3 * x[9] + 2 * x[12] >= minimo[60], "Minimo_60"
model3 += x[2] + x[5] + x[6] + 2 * x[7] + 3 * x[9] + 2 * x[12] <= maximo[60], "Maximo_60"

# Necesidades de Rollos de 70
model3 += x[3] + x[5] + 2 * x[8] + x[11] + x[12] >= minimo[70], "Minimo_70"
model3 += x[3] + x[5] + 2 * x[8] + x[11] + x[12] <= maximo[70], "Maximo_70"

# Necesidades de Rollos de 80
model3 += x[4] + x[6] + 2 * x[10] + x[11] >= minimo[80], "Minimo_80"
model3 += x[4] + x[6] + 2 * x[10] + x[11] <= maximo[80], "Maximo_80"

# Resolver
model3.solve(pulp.PULP_CBC_CMD(msg=False))

# Resultados
print(f"Estado: {pulp.LpStatus[model3.status]}")
print(f"Desperdicio mínimo: {int(pulp.value(model3.objective))} pulgadas\n")

# Patrones cortados
print(f"{'Patrón':<10}{'Rollos':>10}{'Residuo':>10}")
print("-" * 30)
for i in I:
    cant = pulp.value(x[i])
    if cant and cant > 0:
        print(f"P{i:<9}{int(cant):>10}{residuos[i]:>10}")

# Cumplimiento de demanda
print(f"\n{'Ancho':<8}{'Mín':>8}{'Prod':>8}{'Máx':>8}")
print("-" * 32)

produccion_por_ancho = {
    50: 3*x[1] + 2*x[2] + 2*x[3] + 2*x[4] + x[5] + x[6] + x[7] + x[8],
    60: x[2] + x[5] + x[6] + 2*x[7] + 3*x[9] + 2*x[12],
    70: x[3] + x[5] + 2*x[8] + x[11] + 2*x[12],
    80: x[4] + x[6] + 2*x[10] + x[11],
}

for a in anchos:
    prod = int(pulp.value(produccion_por_ancho[a]))
    print(f'{a}"      {minimo[a]:>8}{prod:>8}{maximo[a]:>8}')

Estado: Optimal
Desperdicio mínimo: 30670 pulgadas

Patrón        Rollos   Residuo
------------------------------
P2               967        30
P4               166        10
P6              2834         0
P8              1900         0

Ancho        Mín    Prod     Máx
--------------------------------
50"          7000    7000    8000
60"          3800    3801    4500
70"          3200    3800    3800
80"          2500    3000    3000
